<a href="https://bio-pro.mintlify.app/tools/guides/tool-persistence"><img align="right" src="https://img.shields.io/badge/View_in_Proto_Docs_%E2%86%92-046e7a?style=for-the-badge&logo=readthedocs&logoColor=white" alt="View in Proto Docs: Tool Persistence"></a>

# 02 — Tool Persistence

> **Local inference only.** This notebook covers how `proto_tools` manages models on your own hardware. If you're running via **cloud inference**, this is all handled for you — skip ahead to the next feature.

> Prerequisites: [01 — Getting Started](01_getting_started.ipynb). This notebook assumes you understand the `Input` / `Config` / `run_*()` / `Output` pattern.

When running locally, every tool in `proto_tools` runs in an **isolated virtual environment** via the `ToolInstance` class. This keeps heavy dependencies (PyTorch, ESM, …) out of your main environment.

By default, each `run_*` call spins up a fresh subprocess, loads the model, runs inference, and exits. That's safe and leak-free — but if you're in a batch or optimization loop, you'll reload the model every call. For ESMFold that's ~30 seconds of pure overhead per call.

`ToolInstance` gives you opt-in persistence to fix that:

| Method | Caches? | Cleanup | Best for |
|---|---|---|---|
| Default (one-shot) | No | Automatic | Single calls, safety-first |
| `ToolInstance.persist()` | Yes, auto | Automatic on exit | Batch jobs, optimization loops |
| `ToolInstance.persist_tool()` | Yes, only the named tool | Automatic on exit | Multi-GPU, multiple instances of one tool |
| `ToolInstance.get()` | Yes, until `shutdown()` | Manual | Long-running interactive sessions |

**Device handling.** Device is set via `config.device` (a `BaseConfig` field — GPU tools default to `"cuda"`). The device string flows through to the subprocess and sets `CUDA_VISIBLE_DEVICES`.

**Auto-restart on config changes.** Persistent workers automatically restart when any field marked `reload_on_change=True` changes between calls (e.g. `device`, `model_checkpoint`). No manual reopening needed when you switch models or devices.

**Timeout.** Every call enforces a configurable timeout (default 600 s). Override via `config.timeout`.

We'll demonstrate each mode using ESMFold — you need one GPU to run this notebook.


In [10]:
from time import time

from proto_tools.tools.structure_prediction.esmfold import (
    ESMFoldConfig,
    ESMFoldInput,
    run_esmfold,
)
from proto_tools.utils.tool_instance import ToolInstance

---
## 1. Default behavior (one-shot)

Every `run_*` function uses one-shot execution by default. Each call spins up an ephemeral subprocess, loads the model, runs inference, and exits. No leaked processes, no GPU memory retained.


In [11]:
# Two consecutive calls — each loads the model from scratch
start = time()
output1 = run_esmfold(
    ESMFoldInput(complexes=["MKTLLILAVVAAALA"]),
    ESMFoldConfig(device="cuda"),
)
t1 = time() - start
print(f"First call:  {t1:.1f}s (model load + inference)")

start2 = time()
output2 = run_esmfold(
    ESMFoldInput(complexes=["MKTLLILAVVAAALA"]),
    ESMFoldConfig(device="cuda"),
)
t2 = time() - start2
print(f"Second call: {t2:.1f}s (model load + inference again)")
print(f"Total:       {t1 + t2:.1f}s")

Folding structures (ESMFold):   0%|          | 0/1 [00:00<?, ?structure/s]

First call:  0.6s (model load + inference)


Folding structures (ESMFold):   0%|          | 0/1 [00:00<?, ?structure/s]

Second call: 0.6s (model load + inference again)
Total:       1.2s


---
## 2. `persist()` context manager (recommended)

The simplest way to use persistence. Like `torch.inference_mode()`, just wrap your code — any tool called inside is auto-cached on first use and cleaned up on exit. No tool names needed upfront.


In [12]:
sequences = ["MKTLLILAVVAAALA", "MKTLLILAVVAAALA"]

start = time()
with ToolInstance.persist():
    for i, seq in enumerate(sequences):
        call_start = time()
        output = run_esmfold(
            ESMFoldInput(complexes=[seq]),
            ESMFoldConfig(device="cuda"),
        )
        elapsed = time() - call_start
        label = "(model load + inference)" if i == 0 else "(inference only)"
        print(f"Call {i+1}: {elapsed:.1f}s {label}")

# Block exited — the auto-cached worker has been shut down and GPU memory freed.
print(f"\nTotal: {time() - start:.1f}s")

Folding structures (ESMFold):   0%|          | 0/1 [00:00<?, ?structure/s]

Call 1: 11.4s (model load + inference)


Folding structures (ESMFold):   0%|          | 0/1 [00:00<?, ?structure/s]

Call 2: 0.5s (inference only)

Total: 12.2s


---
## 3. `persist_tool()` — tool-specific persistence

When you need to name a specific tool or manage multiple instances of the same tool (e.g. one per GPU), use `persist_tool()` with an explicit tool name. For a single instance, it works just like `persist()` but scoped to one tool.


In [13]:
sequences = ["MKTLLILAVVAAALA", "MKTLLILAVVAAALA"]

start = time()
# One instance, no instance name needed — dispatch auto-routes to it.
with ToolInstance.persist_tool("esmfold"):
    for i, seq in enumerate(sequences):
        call_start = time()
        output = run_esmfold(
            ESMFoldInput(complexes=[seq]),
            ESMFoldConfig(device="cuda"),
        )
        elapsed = time() - call_start
        label = "(model load + inference)" if i == 0 else "(inference only)"
        print(f"Call {i+1}: {elapsed:.1f}s {label}")

print(f"\nTotal: {time() - start:.1f}s")

Folding structures (ESMFold):   0%|          | 0/1 [00:00<?, ?structure/s]

Call 1: 11.5s (model load + inference)


Folding structures (ESMFold):   0%|          | 0/1 [00:00<?, ?structure/s]

Call 2: 0.6s (inference only)

Total: 12.4s


### Explicit instance passing (multi-instance / multi-GPU)

When you need multiple instances of the same tool running simultaneously — e.g. one per GPU, or different model checkpoints side by side — use named instances and pass `instance=` to route each call to a specific worker.

```python
# Two named instances of the same tool — each gets its own worker process.
# (This cell is illustrative; skip it if you only have one GPU.)
with ToolInstance.persist_tool("esmfold", instance_name="worker_a") as inst_a:
    with ToolInstance.persist_tool("esmfold", instance_name="worker_b"):

        # Route each call to a specific worker
        out_a = run_esmfold(
            ESMFoldInput(complexes=["MKTLLILAVVAAALA"]),
            ESMFoldConfig(device="cuda:0"),
            instance=inst_a,         # pass the instance object
        )
        out_b = run_esmfold(
            ESMFoldInput(complexes=["MKTLLILAVVAAALA"]),
            ESMFoldConfig(device="cuda:1"),
            instance="worker_b",     # or pass the instance name
        )
```


---
## 4. `get()` / `shutdown()` — manual lifecycle

For long-running interactive sessions (e.g. a notebook you'll come back to) where you want to keep a model warm across many cells. You control when the worker is created and destroyed.

Call `tool.shutdown()` when you're done — this stops the worker **and** evicts the instance from the cache.


In [14]:
# get() creates a persistent instance that lives until you explicitly shut it down.
tool = ToolInstance.get("esmfold")

# No `instance=` needed — dispatch auto-finds the cached instance.
for seq in ["MKTLLILAVVAAALA", "GAVLTVLLGGLLLA"]:
    output = run_esmfold(
        ESMFoldInput(complexes=[seq]),
        ESMFoldConfig(device="cuda"),
    )
    print(f"  pLDDT={output.structures[0].metrics.avg_plddt:.2f}")

# shutdown() stops the worker and evicts the instance from the cache.
tool.shutdown()

Folding structures (ESMFold):   0%|          | 0/1 [00:00<?, ?structure/s]

  pLDDT=0.87


Folding structures (ESMFold):   0%|          | 0/1 [00:00<?, ?structure/s]

  pLDDT=0.71


---
## 5. Auto-restart on config changes

Persistent workers automatically restart when any config field marked `reload_on_change=True` changes between calls. This includes `device`, `model_checkpoint`, `model_name`, and other load-time parameters.

You don't pass a device to `persist()` / `persist_tool()` / `get()` — the worker detects config mismatches from the values in your call and restarts transparently.

`device` is a field on `BaseConfig`, so every tool config inherits it. GPU tools default to `"cuda"`; CPU tools default to `"cpu"`. Other `reload_on_change` fields (like `model_checkpoint`) are defined per tool.


---
## 6. Timeout

Every tool call enforces a configurable timeout (default 600 s / 10 minutes). If a call exceeds the timeout, the subprocess is killed and a `TimeoutError` is raised. This works for both one-shot and persistent modes.

```python
# Override the default via config.timeout
output = run_esmfold(
    ESMFoldInput(complexes=["MKTLLILAVVAAALA"]),
    ESMFoldConfig(device="cuda", timeout=120),
)
```

For persistent workers, the timeout applies **per call** — the worker stays alive for future calls even if one call times out.


## What's next?

- **[03 — Device Management](03_device_management.ipynb)** — Pin instances to specific GPUs, understand LRU eviction, pack multiple models on one GPU.
- **[04 — Parallel Execution](04_parallel_execution.ipynb)** — Fan out a batch across all GPUs on the machine.
